In [ ]:


!pip install openai langchain langchain-openai crewai crewai-tools requests -q

In [ ]:


import os
import re
import json
import time
import requests
from datetime import datetime
from typing import Dict, List, Any, Optional
from dataclasses import dataclass
from enum import Enum
from langchain_openai import ChatOpenAI
from langchain.agents import tool, create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

stream_handler = StreamingStdOutCallbackHandler()

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    streaming=True,
    callbacks=[stream_handler],
    request_timeout=600,
    temperature=0
)

print("✓ Modelo configurado con streaming habilitado")
print(f"Modelo: {llm.model_name}")
print(f"Streaming: {llm.streaming}")

In [ ]:

CENTROS = {
    "ensenada": {"lat": -41.140459, "lon": -72.404236, "nombre": "Piscicultura Petrohué"},
    "puelche":  {"lat": -41.733,    "lon": -73.602,    "nombre": "Centro Puelche"},
    "huito":    {"lat": -41.783,    "lon": -73.583,    "nombre": "Centro Huito (San José)"}
}

@tool
def get_clima_actual(centro: str) -> str:
    """Obtiene el clima actual para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."

    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )

    try:
        response = requests.get(url, timeout=10)
        data    = response.json()
        current = data["current"]

        temp      = current["temperature_2m"]
        viento    = current["wind_speed_10m"]
        lluvia    = current["precipitation"]
        codigo    = current["weathercode"]
        condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"

        return (
            f"Centro: {datos['nombre']}\n"
            f"Temperatura: {temp}°C\n"
            f"Viento: {viento} km/h\n"
            f"Precipitación: {lluvia} mm\n"
            f"Condición: {condicion}"
        )
    except Exception as e:
        return f"Error al obtener datos climáticos: {e}"


@tool
def get_pronostico_semana(centro: str) -> str:
    """Obtiene el pronóstico climático de 7 días para un centro de cultivo de Camanchaca.
    El parámetro centro puede ser: ensenada, puelche o huito."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado. Opciones: ensenada, puelche, huito."

    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,"
        f"wind_speed_10m_max,weathercode"
        f"&timezone=America/Santiago"
    )

    try:
        response = requests.get(url, timeout=10)
        data  = response.json()
        daily = data["daily"]

        resultado = f"Pronóstico 7 días - {datos['nombre']}:\n"
        for i in range(7):
            fecha     = daily["time"][i]
            tmax      = daily["temperature_2m_max"][i]
            tmin      = daily["temperature_2m_min"][i]
            lluvia    = daily["precipitation_sum"][i]
            viento    = daily["wind_speed_10m_max"][i]
            codigo    = daily["weathercode"][i]
            condicion = "Despejado" if codigo < 3 else "Nublado" if codigo < 50 else "Lluvia"

            resultado += (
                f"\n{fecha}: {tmin}°C - {tmax}°C | "
                f"Viento: {viento} km/h | "
                f"Lluvia: {lluvia} mm | {condicion}"
            )
        return resultado
    except Exception as e:
        return f"Error al obtener pronóstico: {e}"


@tool
def evaluar_operacion(centro: str, operacion: str) -> str:
    """Evalúa si las condiciones climáticas son seguras para realizar una operación.
    centro: ensenada, puelche o huito.
    operacion: cosecha, biometría o tratamiento."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado."

    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&current=temperature_2m,wind_speed_10m,precipitation,weathercode"
        f"&timezone=America/Santiago"
    )

    try:
        response = requests.get(url, timeout=10)
        data    = response.json()
        current = data["current"]

        viento = current["wind_speed_10m"]
        lluvia = current["precipitation"]
        temp   = current["temperature_2m"]

        alertas = []
        if viento > 40:
            alertas.append(f" Viento peligroso: {viento} km/h (límite: 40 km/h)")
        if lluvia > 10:
            alertas.append(f" Lluvia intensa: {lluvia} mm")
        if temp < 5:
            alertas.append(f" Temperatura muy baja: {temp}°C")
        if temp > 18:
            alertas.append(f" Temperatura elevada: {temp}°C (riesgo para FCR)")

        if not alertas:
            return f" Condiciones APTAS para {operacion} en {datos['nombre']}."
        else:
            return (
                f" Condiciones NO APTAS para {operacion} en {datos['nombre']}:\n"
                + "\n".join(alertas)
            )
    except Exception as e:
        return f"Error al evaluar condiciones: {e}"


@tool
def get_mejor_dia_operacion(centro: str, operacion: str) -> str:
    """Determina el mejor día de la semana para realizar una operación en un centro.
    centro: ensenada, puelche o huito.
    operacion: cosecha, biometría o tratamiento."""
    if centro.lower() not in CENTROS:
        return f"Centro '{centro}' no encontrado."

    datos = CENTROS[centro.lower()]
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={datos['lat']}&longitude={datos['lon']}"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,"
        f"wind_speed_10m_max,weathercode"
        f"&timezone=America/Santiago"
    )

    try:
        response = requests.get(url, timeout=10)
        data  = response.json()
        daily = data["daily"]

        mejor_dia   = None
        mejor_score = -1

        for i in range(7):
            fecha  = daily["time"][i]
            tmax   = daily["temperature_2m_max"][i]
            tmin   = daily["temperature_2m_min"][i]
            lluvia = daily["precipitation_sum"][i]
            viento = daily["wind_speed_10m_max"][i]

            score = 100
            if viento > 40:
                score -= 50
            elif viento > 25:
                score -= 20

            if lluvia > 10:
                score -= 40
            elif lluvia > 5:
                score -= 15

            if tmax > 18:
                score -= 20
            elif tmin < 5:
                score -= 10

            if score > mejor_score:
                mejor_score = score
                mejor_dia   = {
                    "fecha": fecha, "score": score,
                    "tmin": tmin,   "tmax": tmax,
                    "lluvia": lluvia, "viento": viento
                }

        return (
            f"Mejor día para {operacion} en {datos['nombre']}:\n"
            f"Fecha: {mejor_dia['fecha']}\n"
            f"Temperatura: {mejor_dia['tmin']}°C - {mejor_dia['tmax']}°C\n"
            f"Viento máx: {mejor_dia['viento']} km/h\n"
            f"Lluvia acum: {mejor_dia['lluvia']} mm\n"
            f"Puntaje de aptitud: {mejor_dia['score']}/100"
        )
    except Exception as e:
        return f"Error al calcular mejor día: {e}"


tools = [get_clima_actual, get_pronostico_semana, evaluar_operacion, get_mejor_dia_operacion]

print("✓ Herramientas climáticas definidas.")
print(f"  Herramientas: {[t.name for t in tools]}")

In [ ]:


class EstrategiaPlaneamiento(Enum):
    JERARQUICA  = "jerarquica"
    REACTIVA    = "reactiva"
    POR_OBJETIVOS = "por_objetivos"


@dataclass
class PasoOperativo:
    accion:      str
    descripcion: str
    centro:      str
    prioridad:   int   = 1
    duracion_hrs: float = 1.0
    estado:      str   = "pendiente"
    dependencias: List[str] = None

    def __post_init__(self):
        if self.dependencias is None:
            self.dependencias = []


@dataclass
class PlanOperativo:
    objetivo:     str
    pasos:        List[PasoOperativo]
    creado_en:    str
    estado:       str   = "creado"
    duracion_total: float = 0.0

    def __post_init__(self):
        self.duracion_total = sum(p.duracion_hrs for p in self.pasos)


def crear_plan_jerarquico(objetivo: str, centros: List[str]) -> PlanOperativo:
    """Crea un plan jerárquico descomponiendo el objetivo en pasos por centro."""
    print(f"\n Creando plan jerárquico para: {objetivo}")

    pasos = []
    for i, centro in enumerate(centros):
        pasos.append(PasoOperativo(
            accion=f"verificar_clima_{centro}",
            descripcion=f"Verificar condiciones climáticas actuales en {centro}",
            centro=centro,
            prioridad=1,
            duracion_hrs=0.5
        ))
        pasos.append(PasoOperativo(
            accion=f"evaluar_operacion_{centro}",
            descripcion=f"Evaluar viabilidad de operación en {centro}",
            centro=centro,
            prioridad=2,
            duracion_hrs=0.5,
            dependencias=[f"verificar_clima_{centro}"]
        ))
        pasos.append(PasoOperativo(
            accion=f"programar_operacion_{centro}",
            descripcion=f"Programar fecha óptima de operación en {centro}",
            centro=centro,
            prioridad=3,
            duracion_hrs=1.0,
            dependencias=[f"evaluar_operacion_{centro}"]
        ))

    plan = PlanOperativo(
        objetivo=objetivo,
        pasos=pasos,
        creado_en=datetime.now().strftime("%Y-%m-%d %H:%M")
    )

    print(f"  ✓ Plan creado con {len(pasos)} pasos para {len(centros)} centros.")
    print(f"  ✓ Duración estimada: {plan.duracion_total} horas.")
    return plan


def ejecutar_plan(plan: PlanOperativo):
    """Ejecuta un plan operativo mostrando el progreso."""
    print(f"\n Ejecutando plan: {plan.objetivo}")
    print("=" * 50)

    pasos_ordenados = sorted(plan.pasos, key=lambda x: x.prioridad)

    for paso in pasos_ordenados:
        print(f"\n   Paso: {paso.descripcion}")
        print(f"     Centro: {paso.centro} | Prioridad: {paso.prioridad}")

        if paso.dependencias:
            print(f"     Dependencias: {paso.dependencias}")

        time.sleep(0.2)
        paso.estado = "completado"
        print(f"     Estado:  Completado")

    completados = sum(1 for p in plan.pasos if p.estado == "completado")
    plan.estado  = "completado"

    print(f"\n Resumen: {completados}/{len(plan.pasos)} pasos completados.")
    return plan


print("✓ Clases de planificación definidas.")

In [ ]:


print("=== PLANIFICACIÓN JERÁRQUICA CAMANCHACA ===\n")

plan_semanal = crear_plan_jerarquico(
    objetivo="Planificar operaciones semanales en centros de cultivo",
    centros=["ensenada", "puelche", "huito"]
)

ejecutar_plan(plan_semanal)

print("\n=== ESTADO FINAL DEL PLAN ===")
for paso in plan_semanal.pasos:
    icono = "✅" if paso.estado == "completado" else "⏳"
    print(f"{icono} [{paso.centro.upper()}] {paso.descripcion}")

In [ ]:


class PlanificadorReactivo:
    def __init__(self):
        self.reglas  = []
        self.alertas = []

    def agregar_regla(self, condicion, accion, descripcion):
        self.reglas.append({
            "condicion":   condicion,
            "accion":      accion,
            "descripcion": descripcion
        })

    def evaluar_estado(self, estado: Dict[str, Any]) -> List[str]:
        acciones_activadas = []
        for regla in self.reglas:
            if regla["condicion"](estado):
                resultado = regla["accion"](estado)
                acciones_activadas.append(f"[{regla['descripcion']}] → {resultado}")
                self.alertas.append(resultado)
        return acciones_activadas


planificador_reactivo = PlanificadorReactivo()

planificador_reactivo.agregar_regla(
    condicion=lambda s: s.get("viento", 0) > 40,
    accion=lambda s: f" ALERTA: Suspender cosecha en {s.get('centro')} - Viento {s.get('viento')} km/h",
    descripcion="Viento peligroso"
)
planificador_reactivo.agregar_regla(
    condicion=lambda s: s.get("lluvia", 0) > 10,
    accion=lambda s: f" AVISO: Posponer biometría en {s.get('centro')} - Lluvia intensa {s.get('lluvia')} mm",
    descripcion="Lluvia intensa"
)
planificador_reactivo.agregar_regla(
    condicion=lambda s: s.get("temperatura", 15) > 18,
    accion=lambda s: f" ALERTA: Revisar FCR en {s.get('centro')} - Temperatura {s.get('temperatura')}°C",
    descripcion="Temperatura elevada"
)
planificador_reactivo.agregar_regla(
    condicion=lambda s: s.get("temperatura", 15) < 5,
    accion=lambda s: f" AVISO: Monitorear metabolismo en {s.get('centro')} - Temperatura {s.get('temperatura')}°C",
    descripcion="Temperatura muy baja"
)

print("=== PLANIFICACIÓN REACTIVA - SIMULACIÓN DE ALERTAS ===\n")

escenarios = [
    {"centro": "Ensenada", "viento": 55, "lluvia": 2,  "temperatura": 12},
    {"centro": "Puelche",  "viento": 15, "lluvia": 15, "temperatura": 14},
    {"centro": "Huito",    "viento": 20, "lluvia": 3,  "temperatura": 21},
    {"centro": "Ensenada", "viento": 10, "lluvia": 1,  "temperatura": 11},
]

for escenario in escenarios:
    print(f" Estado sensor - {escenario['centro']}:")
    print(f"   Viento: {escenario['viento']} km/h | "
          f"Lluvia: {escenario['lluvia']} mm | "
          f"Temp: {escenario['temperatura']}°C")

    acciones = planificador_reactivo.evaluar_estado(escenario)

    if acciones:
        for accion in acciones:
            print(f"   {accion}")
    else:
        print(f"    Sin alertas - Condiciones normales")
    print()

In [ ]:


prompt_planificador = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres el coordinador de operaciones de Salmones Camanchaca. "
        "Tu rol es planificar las actividades semanales de los centros "
        "Ensenada, Puelche y Huito basándote en las condiciones climáticas "
        "reales obtenidas con tus herramientas. "
        "Siempre consulta el clima antes de tomar decisiones operativas. "
        "Entrega un plan estructurado con días y centros específicos."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agente_planificador = create_openai_tools_agent(llm, tools, prompt_planificador)
executor_planificador = AgentExecutor(
    agent=agente_planificador,
    tools=tools,
    verbose=True
)

print("✓ Agente planificador LangChain listo.")

In [ ]:
print("=== PLANIFICACIÓN SEMANAL CON AGENTE ===\n")

query_plan = (
    "Necesito planificar las operaciones de esta semana. "
    "Tenemos cosecha programada en Ensenada, biometría en Puelche "
    "y tratamiento sanitario en Huito. "
    "¿Cuáles son los mejores días para cada operación según el clima?"
)

response_plan = executor_planificador.invoke({
    "input":        query_plan,
    "chat_history": []
})

print(f"\n Plan Operativo:\n{response_plan['output']}")

In [ ]:


from crewai import Agent, Task, Crew, Process
from crewai_tools import BaseTool

import os
os.environ["OPENAI_API_BASE"] = os.environ.get("OPENAI_BASE_URL", "")
os.environ["OPENAI_API_KEY"]  = os.environ.get("GITHUB_TOKEN", "")

llm_crew = ChatOpenAI(model="gpt-4o", temperature=0)

class ClimaActualTool(BaseTool):
    name: str        = "Clima Actual Camanchaca"
    description: str = "Obtiene el clima actual de un centro. Parámetro: centro (ensenada, puelche, huito)."

    def _run(self, centro: str) -> str:
        return get_clima_actual.invoke(centro)


class PronosticoTool(BaseTool):
    name: str        = "Pronóstico Semanal Camanchaca"
    description: str = "Obtiene el pronóstico 7 días de un centro. Parámetro: centro (ensenada, puelche, huito)."

    def _run(self, centro: str) -> str:
        return get_pronostico_semana.invoke(centro)


class EvaluacionTool(BaseTool):
    name: str        = "Evaluación de Operación Camanchaca"
    description: str = "Evalúa si las condiciones son aptas. Parámetros: centro y operacion."

    def _run(self, centro: str, operacion: str = "cosecha") -> str:
        return evaluar_operacion.invoke({"centro": centro, "operacion": operacion})


class MejorDiaTool(BaseTool):
    name: str        = "Mejor Día para Operación Camanchaca"
    description: str = "Determina el mejor día de la semana para una operación. Parámetros: centro y operacion."

    def _run(self, centro: str, operacion: str = "cosecha") -> str:
        return get_mejor_dia_operacion.invoke({"centro": centro, "operacion": operacion})


clima_tool    = ClimaActualTool()
pronostico_tool = PronosticoTool()
evaluacion_tool = EvaluacionTool()
mejor_dia_tool  = MejorDiaTool()

print("✓ Herramientas CrewAI definidas.")

In [ ]:
meteorologo_crew = Agent(
    role="Meteorólogo Acuícola Senior",
    goal=(
        "Analizar en detalle las condiciones climáticas actuales y el pronóstico semanal "
        "de los tres centros de cultivo de Camanchaca: Ensenada, Puelche y Huito."
    ),
    backstory=(
        "Eres un meteorólogo con 15 años de experiencia en el clima marítimo del sur de Chile. "
        "Conoces perfectamente los patrones climáticos de la región de Los Lagos y su impacto "
        "en las operaciones acuícolas. Tu análisis es la base de todas las decisiones operativas."
    ),
    tools=[clima_tool, pronostico_tool],
    llm=llm_crew,
    verbose=True,
    allow_delegation=False
)

coordinador_crew = Agent(
    role="Coordinador de Operaciones Camanchaca",
    goal=(
        "Planificar y optimizar el calendario operativo semanal de los centros de cultivo "
        "basándose en el análisis climático, priorizando seguridad y eficiencia productiva."
    ),
    backstory=(
        "Eres el coordinador de operaciones de Salmones Camanchaca con experiencia en "
        "la planificación de cosechas, biometrías y tratamientos sanitarios. "
        "Conoces los protocolos de seguridad y los límites operativos de cada actividad."
    ),
    tools=[evaluacion_tool, mejor_dia_tool],
    llm=llm_crew,
    verbose=True,
    allow_delegation=False
)

supervisor_crew = Agent(
    role="Supervisor General Camanchaca",
    goal=(
        "Revisar el plan operativo, identificar riesgos y emitir el reporte final "
        "con recomendaciones para la gerencia de Salmones Camanchaca."
    ),
    backstory=(
        "Eres el supervisor general con visión integral del negocio acuícola. "
        "Tu responsabilidad es validar que el plan operativo sea seguro, viable "
        "y alineado con los objetivos productivos de la empresa."
    ),
    tools=[evaluacion_tool],
    llm=llm_crew,
    verbose=True,
    allow_delegation=False
)

print("✓ Equipo CrewAI definido:")
print(f"   {meteorologo_crew.role}")
print(f"   {coordinador_crew.role}")
print(f"   {supervisor_crew.role}")

In [ ]:
tarea_clima = Task(
    description=(
        "Realiza un análisis climático completo de los tres centros de Camanchaca: "
        "Ensenada, Puelche y Huito. Para cada centro obtén el clima actual y el pronóstico "
        "de 7 días. Identifica los días de mayor riesgo por viento, lluvia o temperatura "
        "extrema que puedan afectar las operaciones."
    ),
    expected_output=(
        "Reporte climático detallado por centro con: condiciones actuales, "
        "pronóstico semanal y días críticos identificados."
    ),
    agent=meteorologo_crew
)

tarea_planificacion = Task(
    description=(
        "Basándote en el análisis climático, planifica el calendario operativo semanal. "
        "Determina el mejor día para: cosecha en Ensenada, biometría en Puelche y "
        "tratamiento sanitario en Huito. Justifica cada decisión con datos climáticos."
    ),
    expected_output=(
        "Calendario operativo semanal con días asignados para cada actividad, "
        "condiciones esperadas y justificación técnica."
    ),
    agent=coordinador_crew,
    context=[tarea_clima]
)

tarea_reporte = Task(
    description=(
        "Revisa el plan operativo propuesto y elabora el reporte final para la gerencia. "
        "Incluye: resumen ejecutivo, plan aprobado, riesgos identificados y "
        "recomendaciones de contingencia si las condiciones cambian."
    ),
    expected_output=(
        "Reporte ejecutivo en formato estructurado con resumen, plan aprobado, "
        "riesgos y recomendaciones de contingencia."
    ),
    agent=supervisor_crew,
    context=[tarea_clima, tarea_planificacion]
)

crew_camanchaca = Crew(
    agents=[meteorologo_crew, coordinador_crew, supervisor_crew],
    tasks=[tarea_clima, tarea_planificacion, tarea_reporte],
    process=Process.sequential,
    verbose=True
)

print("\n✓ Crew Camanchaca ensamblado.")
print(f"  Agentes: {len(crew_camanchaca.agents)}")
print(f"  Tareas:  {len(crew_camanchaca.tasks)}")
print(f"  Proceso: {crew_camanchaca.process}")

In [ ]:


try:
    print(" Iniciando equipo de planificación Camanchaca...\n")
    resultado = crew_camanchaca.kickoff()

    print("\n" + "=" * 60)
    print(" REPORTE FINAL - SALMONES CAMANCHACA")
    print("=" * 60)
    print(resultado)

except Exception as e:
    print(f" Error: {e}")
    import traceback
    traceback.print_exc()

In [ ]:


print("=== COMPARACIÓN DE ESTRATEGIAS DE PLANIFICACIÓN ===\n")

comparacion = {
    "Jerárquica": {
        "Descripción":  "Descompone el objetivo en subpasos ordenados por prioridad.",
        "Ventajas":     "Estructura clara, fácil de seguir y auditar.",
        "Desventajas":  "Poco flexible ante cambios inesperados del clima.",
        "Uso en Camanchaca": "Planificación semanal rutinaria de operaciones."
    },
    "Reactiva": {
        "Descripción":  "Responde en tiempo real a cambios del entorno (alertas climáticas).",
        "Ventajas":     "Inmediata, adaptable a condiciones cambiantes del sur de Chile.",
        "Desventajas":  "Puede ser impredecible, sin visión a largo plazo.",
        "Uso en Camanchaca": "Sistema de alertas por viento, lluvia o temperatura."
    },
    "Por Objetivos (CrewAI)": {
        "Descripción":  "Múltiples agentes especializados colaboran para alcanzar un objetivo.",
        "Ventajas":     "Análisis integral, cada agente aporta su especialidad.",
        "Desventajas":  "Mayor costo computacional y tiempo de ejecución.",
        "Uso en Camanchaca": "Reporte ejecutivo semanal para la gerencia."
    }
}

for estrategia, info in comparacion.items():
    print(f" {estrategia}:")
    for clave, valor in info.items():
        print(f"   {clave}: {valor}")
    print()